# Agent的高级用法

## 1、设置agent的名称

使用字段name来进行设置

举例：

In [1]:
import truststore
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
from rich import print as rprint


load_dotenv(override=True)
truststore.inject_into_ssl()

# 以init_chat_model为例
# model = init_chat_model(
#     model="gpt-5.4-mini",
#     model_provider="openai",
#     api_key=os.getenv("CLOSEAI_API_KEY"),
#     base_url=os.getenv("CLOSEAI_BASE_URL")
# )
model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    extra_body={"thinking": {"type": "disabled"}},
)


agent = create_agent(
    model=model,
    name="agent01"
)

# 调用
response= agent.invoke({
    "messages":[
        {"role":"system","content":"你是一个精通数学的老师，擅长以通俗易懂的方式讲解数学问题"},
        {"role":"user","content":"100 + 20 * 3 = ？"}
    ]
})


# rprint(response)

for message in response["messages"]:
    message.pretty_print()

================================ System Message ================================

你是一个精通数学的老师，擅长以通俗易懂的方式讲解数学问题
================================ Human Message =================================

100 + 20 * 3 = ？
================================== Ai Message ==================================
Name: agent01

这个问题按照数学的**运算规则**（先乘除，后加减）来计算：

1. **先算乘法**：\( 20 \times 3 = 60 \)  
2. **再算加法**：\( 100 + 60 = 160 \)

所以答案是 **160**。

简单记个口诀：**“先乘除，后加减”**，就像出门要穿鞋再戴帽，顺序不能乱哦！😊


## 2、系统提示词的设置

使用system_prompt参数进行设置，可以是str,也可以是SystemMessage

举例1：

In [ ]:
from langchain_tavily import TavilySearch
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
load_dotenv(override=True)


# 1.导入模型
# model = init_chat_model(
#     model="gpt-5.4-mini",
#     model_provider="openai",
#     api_key=os.getenv("CLOSEAI_API_KEY"),
#     base_url=os.getenv("CLOSEAI_BASE_URL")
# )
model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    extra_body={"thinking": {"type": "disabled"}},
)

# 2.导入工具
web_search = TavilySearch(max_results=2)

# 3.创建Agent
agent = create_agent(
    model=model,
    tools=[web_search],
    system_prompt="你是一名多才多艺的智能助手，可以调用工具帮助用户解决问题。"
)

# 4.运行Agent获得结果
result = agent.invoke(
    {"messages": [
        {"role": "user", "content": "请帮我查询2026年足球世界杯是哪个国家举办的？"}
    ]}
)

rprint(result)

举例2：

In [ ]:
from langchain.agents import create_agent
from langchain_core.messages import SystemMessage
from langchain_core.tools import tool
from rich import print as rprint

# 工具：实现两数相加
@tool
def add_numbers(a: int, b: int) -> str:
    """计算并返回两个数的和。"""
    return f"和为：{a + b}"


# 创建客服助手Agent
agent = create_agent(
    model=model,
    tools=[add_numbers],  # 工具列表
    # system_prompt="你是一个数学助手，解决日常的算术问题"
    system_prompt=SystemMessage(content="你是一个数学助手，解决日常的算术问题")
)

response = agent.invoke(
    {"messages": [
        {"role": "user", "content": "10加上20再加上30是多少？"}
    ]},
)

rprint(response)
# print(response["messages"][-1].content)

举例3：

In [5]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain.messages import SystemMessage, HumanMessage

flag = 0

@tool
def get_weather(city: str):
    """
    天气查询工具

    Args:
        city: 城市名称
    """
    global flag
    flag += 1

    if flag < 3:
        # raise Exception("暂时无法访问")
        return "TEMP_UNAVAILABLE: 天气服务暂时不可用，请稍后重试"

    return f"{city}今天天气挺好"


messages = [
    HumanMessage("你好，杭州今天的天气如何？")
]

agent = create_agent(
    model=model,
    tools=[get_weather],
    system_prompt=SystemMessage(
        "你是一个天气助手。"
        "当工具返回以 'TEMP_UNAVAILABLE:' 开头的结果时，"
        "说明是临时故障，不要立即放弃；"
        "你应再次调用同一个工具，最多重试 3 次。"
        "如果 3 次后仍失败，再向用户说明服务暂时不可用。"
    )
)
response = agent.invoke({"messages": messages})
# print(response)
for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

你好，杭州今天的天气如何？
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_Sv4KotTGb5c1aiUOhFsCzlao)
 Call ID: call_Sv4KotTGb5c1aiUOhFsCzlao
  Args:
    city: 杭州
================================= Tool Message =================================
Name: get_weather

TEMP_UNAVAILABLE: 天气服务暂时不可用，请稍后重试
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_MNlT2p9GrmF9gHaKglDY1flh)
 Call ID: call_MNlT2p9GrmF9gHaKglDY1flh
  Args:
    city: 杭州
================================= Tool Message =================================
Name: get_weather

TEMP_UNAVAILABLE: 天气服务暂时不可用，请稍后重试
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_dOyfcXs0f9K6rDgMI1hiMYLm)
 Call ID: call_dOyfcXs0f9K6rDgMI1hiMYLm
  Args:
    city: 杭州
================================= To

## 3、结构化输出的4种策略

### 3.1 ProviderStrategy策略：

In [3]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.agents.structured_output import ProviderStrategy
from langchain.messages import HumanMessage
from rich import print as rprint
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

load_dotenv(override=True)

# 1.模型初始化
# model = init_chat_model(
#     model="gpt-5.4-mini",
#     model_provider="openai",
#     api_key=os.getenv("CLOSEAI_API_KEY"),
#     base_url=os.getenv("CLOSEAI_BASE_URL")
# )
model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    extra_body={"thinking": {"type": "disabled"}},
)

# 2.使用Pydantic结构化方式定义
class ContractInfo(BaseModel):
    """用户的联系方式"""
    name : str = Field(description="用户的姓名")
    email : str = Field(description="用户的邮箱")
    phone : str = Field(description="用户的电话")

agent = create_agent(
    model = model,
    response_format=ProviderStrategy(ContractInfo)
)


response = agent.invoke({
    "messages": [
        {"role":"user","content":"从以下信息中提取用户信息，小明的邮箱是shkstart@atguigu.com,电话是13012341234"}
    ]
})

rprint(response)

{
    'messages': [
        HumanMessage(
            content='从以下信息中提取用户信息，小明的邮箱是shkstart@atguigu.com,电话是13012341234',
            additional_kwargs={},
            response_metadata={},
            id='c76c17a2-3e32-4eb4-bdbf-411d8ac29cea'
        ),
        AIMessage(
            content='{"name":"小明","email":"shkstart@atguigu.com","phone":"13012341234"}',
            additional_kwargs={'parsed': None, 'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 31,
                    'prompt_tokens': 242,
                    'total_tokens': 273,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0},
                    'latency_checkpoint': {
                        'engine_tbt_ms': 5,
                        'engine_ttft_ms': 30,
                        'engine_ttlt_ms': 183,
                        'pre_inference_ms': 71,
                        'service_tbt_ms': 5,
                        'service_ttft_ms': 254,
                        'service_ttlt_ms': 392,
                        'total_duration_ms': 333,
                        'user_visible_ttft_ms': 183
                    }
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5.4-mini-2026-03-17',
                'system_fingerprint': None,
                'id': 'chatcmpl-DmtjPEaEDe1aUSY1RYSlCW4YTciqH',
                'service_tier': 'default',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--019e90d5-18fe-76d3-af7d-68d77aea40a9-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 242,
                'output_tokens': 31,
                'total_tokens': 273,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        )
    ],
    'structured_response': ContractInfo(name='小明', email='shkstart@atguigu.com', phone='13012341234')
}

### 3.2 ToolStrategy策略：

In [5]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy
from langchain.messages import HumanMessage
from rich import print as rprint
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

load_dotenv(override=True)

# 1.模型初始化
# model = init_chat_model(
#     model="gpt-5.4-mini",
#     model_provider="openai",
#     api_key=os.getenv("CLOSEAI_API_KEY"),
#     base_url=os.getenv("CLOSEAI_BASE_URL")
# )
model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    extra_body={"thinking": {"type": "disabled"}},
)

# 2.使用Pydantic结构化方式定义
class ContractInfo(BaseModel):
    """用户的联系方式"""
    name : str = Field(description="用户的姓名")
    email : str = Field(description="用户的邮箱")
    phone : str = Field(description="用户的电话")

agent = create_agent(
    model = model,
    response_format=ToolStrategy(ContractInfo)
)


response = agent.invoke({
    "messages": [
        # {"role":"user","content":"从以下信息中提取用户信息，小明的邮箱是shkstart@atguigu.com,电话是13012341234"}
        HumanMessage(content="从以下信息中提取用户信息，小明的邮箱是shkstart@atguigu.com,电话是13012341234")
    ]
})

rprint(response)

# print(response["structured_response"])

{
    'messages': [
        HumanMessage(
            content='从以下信息中提取用户信息，小明的邮箱是shkstart@atguigu.com,电话是13012341234',
            additional_kwargs={},
            response_metadata={},
            id='3feca91d-5782-4ed7-b298-83fbccafbf42'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 36,
                    'prompt_tokens': 170,
                    'total_tokens': 206,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0},
                    'latency_checkpoint': {
                        'engine_tbt_ms': 3,
                        'engine_ttft_ms': 33,
                        'engine_ttlt_ms': 134,
                        'pre_inference_ms': 74,
                        'service_tbt_ms': 3,
                        'service_ttft_ms': 254,
                        'service_ttlt_ms': 354,
                        'total_duration_ms': 297,
                        'user_visible_ttft_ms': 180
                    }
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5.4-mini-2026-03-17',
                'system_fingerprint': None,
                'id': 'chatcmpl-DmtnSxyNdNiVaXjWKJPslD8mBkgYU',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019e90d8-f129-7d33-be96-94d7902cfa46-0',
            tool_calls=[
                {
                    'name': 'ContractInfo',
                    'args': {'name': '小明', 'email': 'shkstart@atguigu.com', 'phone': '13012341234'},
                    'id': 'call_nwXQR1YHUJDp2eARMiilsNTR',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 170,
                'output_tokens': 36,
                'total_tokens': 206,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        ),
        ToolMessage(
            content="Returning structured response: name='小明' email='shkstart@atguigu.com' phone='13012341234'",
            name='ContractInfo',
            id='59bd58a0-b0e6-407a-8b79-fced3343e2c1',
            tool_call_id='call_nwXQR1YHUJDp2eARMiilsNTR'
        )
    ],
    'structured_response': ContractInfo(name='小明', email='shkstart@atguigu.com', phone='13012341234')
}

### 3.3 AutoStrategy / Type策略：

In [5]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy, AutoStrategy
from langchain.messages import HumanMessage
from rich import print as rprint
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

load_dotenv(override=True)

# 1.模型初始化
# model = init_chat_model(
#     model="gpt-5.4-mini",
#     model_provider="openai",
#     api_key=os.getenv("CLOSEAI_API_KEY"),
#     base_url=os.getenv("CLOSEAI_BASE_URL")
# )
model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    extra_body={"thinking": {"type": "disabled"}},
)

# 2.使用Pydantic结构化方式定义
class ContractInfo(BaseModel):
    """用户的联系方式"""
    name : str = Field(description="用户的姓名")
    email : str = Field(description="用户的邮箱")
    phone : str = Field(description="用户的电话")

agent = create_agent(
    model = model,
    response_format=AutoStrategy(ContractInfo)
    # response_format=ContractInfo  # 不推荐大家使用
)


response = agent.invoke({
    "messages": [
        # {"role":"user","content":"从以下信息中提取用户信息，小明的邮箱是shkstart@atguigu.com,电话是13012341234"}
        HumanMessage(content="从以下信息中提取用户信息，小明的邮箱是shkstart@atguigu.com,电话是13012341234")
    ]
})

rprint(response)

print(response["structured_response"], type(response["structured_response"]))

{
    'messages': [
        HumanMessage(
            content='从以下信息中提取用户信息，小明的邮箱是shkstart@atguigu.com,电话是13012341234',
            additional_kwargs={},
            response_metadata={},
            id='04094cdc-259e-47ba-9db5-9c4a3212f528'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 78,
                    'prompt_tokens': 347,
                    'total_tokens': 425,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256},
                    'prompt_cache_hit_tokens': 256,
                    'prompt_cache_miss_tokens': 91
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402',
                'id': '21ae3b06-546b-42ad-9bd3-4eda5d01c3ad',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019f7513-1db3-7960-b3c0-13840f274c35-0',
            tool_calls=[
                {
                    'name': 'ContractInfo',
                    'args': {'name': '小明', 'email': 'shkstart@atguigu.com', 'phone': '13012341234'},
                    'id': 'call_00_9GJMirKFWA7zBfQxi2NC0815',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 347,
                'output_tokens': 78,
                'total_tokens': 425,
                'input_token_details': {'cache_read': 256},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content="Returning structured response: name='小明' email='shkstart@atguigu.com' phone='13012341234'",
            name='ContractInfo',
            id='ab2c2036-62ed-4b1c-bd41-542401a7efde',
            tool_call_id='call_00_9GJMirKFWA7zBfQxi2NC0815'
        )
    ],
    'structured_response': ContractInfo(name='小明', email='shkstart@atguigu.com', phone='13012341234')
}

name='小明' email='shkstart@atguigu.com' phone='13012341234' <class '__main__.ContractInfo'>
